# Optimizing an AI Skill — profile-bio Before & After
### Companion notebook for *How Resourceful Is Your AI Skill?* (Part 3)

**→ [Read the full series: How Resourceful Is Your AI Skill?](https://sriharshacr.github.io/blogs/how-resourceful-is-your-ai-skill/)**

---

This notebook walks through the three production gaps identified in [`profile-bio`](https://github.com/SriharshaCR/open-skills/tree/main/profile-bio) — a real AI skill I built and published — and shows the before/after delta for each one:

| Gap | Fix | What you measure |
|---|---|---|
| No token budget | Single-platform generation path | Token count: full run vs targeted run |
| No data handling policy | Typed brief with sensitivity labels | Schema: untyped dict vs declared dataclass |
| No regression suite | Golden test harness | Pass/fail: char limits + tone alignment |

You can read the post without running this. This is for those who want to see it live.

**Connecting Dots**, is where I write about the patterns I notice while building, checkout my blogs for more

👉 https://sriharshacr.github.io/blogs/

## Before you run anything — read this

### Step 1: Create a free Groq account
1. Go to [console.groq.com](https://console.groq.com) and sign up — **free, no credit card required**
2. Navigate to **API Keys** in the left sidebar
3. Click **Create API Key** → give it a name → copy the key

### Step 2: Add your key to Colab Secrets (not to a code cell)
1. In this notebook, click the **🔑 key icon** in the left sidebar
2. Click **+ Add new secret**
3. Name: `GROQ_API_KEY` (exact spelling)
4. Value: paste your API key
5. Toggle **Notebook access** to ON

> ⚠️ **Never paste your API key directly into a code cell.**
> If a notebook containing a key is committed to GitHub — even in a private repo — automated scanners will find it within minutes. Always use Colab Secrets.

---

### A note on model outputs

> ⚠️ LLMs are non-deterministic — the same prompt can produce different outputs each time. If your character counts differ slightly from mine, re-run the cell. The patterns hold even when exact numbers vary.

In [ ]:
%pip install openai --quiet

In [ ]:
# ── Provider config ───────────────────────────────────────────────────────────
# Default: Groq (free, no credit card). To switch providers, update BASE_URL + API_KEY.
#
# OpenAI:  BASE_URL = "https://api.openai.com/v1"   secret: OPENAI_API_KEY
# Kimi:    BASE_URL = "https://api.moonshot.cn/v1"  secret: KIMI_API_KEY
#
BASE_URL = "https://api.groq.com/openai/v1"
#
# ── Model options on Groq (free tier) ────────────────────────────────────────
# llama-3.1-8b-instant     → Fastest (~560 tokens/sec). Good for quick runs.
# llama-3.3-70b-versatile  → Stronger reasoning. Better before/after contrast.
#
MODEL = "llama-3.3-70b-versatile"
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import userdata
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=userdata.get("GROQ_API_KEY"))
print(f"Client ready. Model: {MODEL}")

In [ ]:
import json

def ask(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return {
        "output":        response.choices[0].message.content,
        "input_tokens":  response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
        "total_tokens":  response.usage.prompt_tokens + response.usage.completion_tokens,
    }

def build_prompt(brief, platforms):
    platform_str = ", ".join(platforms)
    return f"""You are a professional bio writer. Generate ready-to-paste bios for: {platform_str}.

Profile brief:
- Name: {brief['name']}
- What they do: {brief['what_you_do']}
- Core belief: {brief['core_belief']}
- Personal interests: {', '.join(brief['personal_texture'])}
- Tone: {brief['tone']}
- CTA: {brief['cta']}
- Links: {', '.join(brief['links'])}

For each platform output exactly one line in this format:
[Platform]: <bio text only — no extra commentary>
Strictly enforce character limits: Twitter/X = 160 chars, LinkedIn headline = 220 chars, GitHub = 160 chars."""

# Fixed test brief — used throughout all experiments for consistent comparison
TEST_BRIEF = {
    "name": "Alex Rivera",
    "what_you_do": "Cloud engineer building AI-native internal tools at a mid-sized fintech",
    "core_belief": "The best AI tools are the ones that disappear into the workflow",
    "personal_texture": ["trail running", "open source contributor", "amateur radio operator"],
    "tone": "sharp & direct",
    "cta": "DM me if you're hiring or building in this space",
    "links": ["https://github.com/alexrivera", "https://alexrivera.dev"],
}

print("Helpers ready. TEST_BRIEF loaded.")

---

## Gap 1: No Token Budget

The default `profile-bio` invocation — no mode flag — runs gather then generates bios for **all 6 platforms** in a single session. Someone who only needed a LinkedIn update still triggers the entire pipeline.

There's no declared token budget. No single-platform path. No definition of what "this run cost more than expected" even means.

The fix: expose a generate-single-platform path. Let the token cost match the actual need.

**Experiment:** Run the full-platform prompt and a single-platform prompt with the same brief. Compare token counts.

In [ ]:
# BEFORE — profile-bio (no mode): generates all 6 platforms
prompt_all = build_prompt(
    TEST_BRIEF,
    ["Twitter/X", "LinkedIn", "GitHub", "Reddit", "Instagram", "YouTube"]
)
before = ask(prompt_all)

print("=== BEFORE — all 6 platforms ===")
print(before["output"])
print(f"\nTotal tokens: {before['total_tokens']} "
      f"(input: {before['input_tokens']}, output: {before['output_tokens']})")

In [ ]:
# AFTER — profile-bio --generate linkedin: one platform, one call
prompt_one = build_prompt(TEST_BRIEF, ["LinkedIn"])
after_token = ask(prompt_one)

print("=== AFTER — LinkedIn only ===")
print(after_token["output"])
print(f"\nTotal tokens: {after_token['total_tokens']} "
      f"(input: {after_token['input_tokens']}, output: {after_token['output_tokens']})")

saved = before["total_tokens"] - after_token["total_tokens"]
reduction = round((1 - after_token["total_tokens"] / before["total_tokens"]) * 100)
print(f"\nDelta: {saved} tokens saved — {reduction}% reduction for a single-platform targeted call")

### What just happened

The model didn't get smarter. The prompt got smaller.

When the user only needs LinkedIn, the full-platform prompt is pure waste — the output tokens for 5 platforms the user didn't ask for are never used. The fix isn't clever; it's just a path that doesn't exist today.

This is what "no declared budget" looks like in practice: **you can't even state whether a run was expensive or cheap**, because there's no reference point. A declared budget (e.g. "full run: ~1,500 tokens; single-platform: ~300 tokens") makes every invocation auditable.

---

## Gap 2: No Data Handling Policy

After gather mode, `profile-bio` writes a `profile-brief.md` to disk. The intent is reuse — run generate again later without repeating the interview.

The brief contains: your name, core beliefs, CTA, personal texture, and links. In a multi-skill agent where other skills have filesystem access, that file is readable by any tool. There's no declared schema, no sensitivity labels, no TTL, and no stated access restriction.

**Experiment:** Show what the brief looks like today vs what a typed brief with declared sensitivity looks like.

In [ ]:
# BEFORE — Level 1: untyped dict written to disk, no policy
brief_before = {
    "name": "Alex Rivera",
    "core_belief": "The best AI tools are the ones that disappear into the workflow",
    "personal_texture": ["trail running", "open source contributor", "amateur radio operator"],
    "cta": "DM me if you're hiring or building in this space",
    "links": ["https://github.com/alexrivera", "https://alexrivera.dev"],
}

print("BEFORE — what profile-brief.md contains today:")
print(json.dumps(brief_before, indent=2))
print()
print("Sensitivity labels  : none")
print("TTL policy          : none declared")
print("Access restriction  : none — readable by any tool with filesystem access")
print("Forwarding policy   : none — any skill can pass this to an external API")

In [ ]:
from dataclasses import dataclass, field
from typing import List, Literal

@dataclass
class ProfileBrief:
    """
    Classification: PRIVATE
    Storage:        Local filesystem only. Do not sync to cloud or pass to external APIs.
    TTL:            30 days from creation. After TTL: delete file, re-run gather.
    Access policy:  Readable only by profile-bio skill.
                    Multi-skill agents MUST NOT read this file without explicit user approval.
    """
    name:             str                                                          # PUBLIC
    what_you_do:      str                                                          # PUBLIC
    cta:              str                                                          # PUBLIC
    links:            List[str] = field(default_factory=list)                     # PUBLIC
    tone:             Literal["warm & witty", "sharp & direct", "calm & thoughtful"] = "sharp & direct"  # PUBLIC
    core_belief:      str = ""                                                     # PRIVATE — do not forward
    personal_texture: List[str] = field(default_factory=list)                     # PRIVATE — do not forward

brief_after = ProfileBrief(
    name=TEST_BRIEF["name"],
    what_you_do=TEST_BRIEF["what_you_do"],
    cta=TEST_BRIEF["cta"],
    links=TEST_BRIEF["links"],
    tone=TEST_BRIEF["tone"],
    core_belief=TEST_BRIEF["core_belief"],
    personal_texture=TEST_BRIEF["personal_texture"],
)

print("AFTER — typed brief with declared sensitivity:")
print(brief_after)
print()
print("Sensitivity labels  : per-field (PUBLIC / PRIVATE)")
print("TTL policy          : 30 days")
print("Access restriction  : profile-bio only")
print("Forwarding policy   : PRIVATE fields must not be passed to external APIs")

### What just happened

The data didn't change. The contract around it did.

A typed brief with sensitivity labels does three things an untyped dict cannot:
1. **It's auditable** — any code that touches the brief knows which fields are sensitive and which aren't
2. **It's enforceable** — a multi-skill agent framework can inspect the docstring policy and refuse to pass `core_belief` to an external API
3. **It decays cleanly** — a TTL makes "how long does my personal data sit on disk?" a question with an answer

One sentence in the spec — "do not forward PRIVATE fields to external APIs" — doesn't exist in `profile-bio` today. That sentence is the gap.

---

## Gap 3: No Behavioral Regression Suite

`profile-bio` depends on the model to enforce three behavioral contracts: interpret tone options consistently, enforce per-platform character limits, and maintain voice across all platforms in a single run.

I validated this once, against one model version. If the model's interpretation of "sharp & direct" shifts in a future update — or if character limit enforcement regresses — the output degrades silently. No test would catch it before a user reports it.

**Experiment:** Run the skill against the same brief, first without validation (eyeball only), then with a golden test harness that checks the behavioral contracts programmatically.

In [ ]:
# BEFORE — Level 1: run once, eyeball the output
result = ask(build_prompt(TEST_BRIEF, ["Twitter/X"]))
raw = result["output"].strip()

# Extract the bio text after the [Platform]: prefix
bio = raw.split(":", 1)[-1].strip() if ":" in raw else raw

print("=== BEFORE — Twitter/X bio (no validation) ===")
print(raw)
print(f"\nCharacter count: {len(bio)} / 160 limit — checked: manually")
print("Tone check: eyeballed — no programmatic verification")

In [ ]:
# AFTER — golden test harness: fixed inputs, programmatic contract verification

CONTRACTS = {
    "Twitter/X": {
        "max_chars": 160,
        "tone_signals": {
            "sharp & direct": ["builds", "works", "ships", "direct", "engineer", "tools"],
        },
    },
    "LinkedIn": {
        "max_chars": 220,
        "tone_signals": {
            "sharp & direct": ["engineer", "builds", "cloud", "fintech", "tools"],
        },
    },
    "GitHub": {
        "max_chars": 160,
        "tone_signals": {
            "sharp & direct": ["engineer", "builds", "open source", "tools"],
        },
    },
}

report = []
for platform, rules in CONTRACTS.items():
    result = ask(build_prompt(TEST_BRIEF, [platform]))
    raw    = result["output"].strip()
    bio    = raw.split(":", 1)[-1].strip() if ":" in raw else raw

    char_ok  = len(bio) <= rules["max_chars"]
    tone_kws = rules["tone_signals"].get(TEST_BRIEF["tone"], [])
    tone_ok  = any(kw.lower() in bio.lower() for kw in tone_kws)

    report.append({
        "platform": platform,
        "bio":       bio,
        "chars":     len(bio),
        "limit":     rules["max_chars"],
        "char_pass": char_ok,
        "tone_pass": tone_ok,
        "verdict":   "PASS" if char_ok and tone_ok else "FAIL",
    })

print("=== AFTER — regression report ===")
print(f"{'Platform':<12} | {'Chars':>8} | {'Char limit':>10} | {'Char':>6} | {'Tone':>6} | Verdict")
print("-" * 68)
for r in report:
    print(f"{r['platform']:<12} | {r['chars']:>8} | {r['limit']:>10} | "
          f"{'✓' if r['char_pass'] else '✗':>6} | "
          f"{'✓' if r['tone_pass'] else '✗':>6} | {r['verdict']}")
print()
for r in report:
    print(f"[{r['platform']}] {r['bio']}")
    print()

### What just happened

The test harness doesn't validate the *content* — it validates the *behavioral contract*: did the model stay within the character limit, and does the output carry tone signals consistent with the selected tone?

A **FAIL on char limit** is the point: it reveals that the model's counting behavior is unreliable, and the skill had no way to surface that before. If you re-run this cell after a model update and the result changes, that's a regression — caught before a user reports it.

A **FAIL on tone** means the generated bio doesn't carry the expected signal for the chosen style. It doesn't mean the bio is bad — it means the tone contract is weaker than assumed.

> The test doesn't need to compare outputs word-for-word. It needs to verify the behavioral contract held.

---

## Delta Summary

Three gaps. Three fixes. Here's what the numbers say.

In [ ]:
char_pass_count = sum(1 for r in report if r["char_pass"])
tone_pass_count = sum(1 for r in report if r["tone_pass"])
n = len(report)
reduction = round((1 - after_token["total_tokens"] / before["total_tokens"]) * 100)

print(f"""
┌─────────────────────────┬────────────────────────┬──────────────────────────────────┐
│ Metric                  │ Before (Level 1)       │ After (Level 2)                  │
├─────────────────────────┼────────────────────────┼──────────────────────────────────┤
│ Token cost (full run)   │ {before['total_tokens']:<22} │ {after_token['total_tokens']:<32} │
│ Token reduction         │ baseline               │ ~{reduction}% per targeted call           │
│ Brief schema            │ Untyped dict           │ Typed dataclass + sensitivity    │
│ Data handling policy    │ None declared          │ TTL 30d, no external forwarding  │
│ Char limit testing      │ Manual / none          │ Automated: {char_pass_count}/{n} platforms pass       │
│ Tone alignment testing  │ None                   │ Automated: {tone_pass_count}/{n} platforms pass       │
└─────────────────────────┴────────────────────────┴──────────────────────────────────┘
""")

### Maturity re-assessment

Before these changes, `profile-bio` sat firmly at **Level 1 — Packaged Skill**: metadata, usage instructions, happy-path examples.

These three fixes move it toward **Level 2 — Verified Skill**:
- Typed boundaries (brief schema)
- Security posture (sensitivity labels, access policy)
- Behavioral tests (regression harness for char limits + tone)
- Token benchmark (declared budget, targeted path)

What's still missing for a full Level 2 certification: a permission manifest (exactly which tools the skill is allowed to call), a signed release, and observability hooks. Those are the backlog items — Part 2 of this series covers what the full contract looks like.

```
Level 0 — Prompt Snippet     No owner, no schema, no tests, no permission scope
Level 1 — Packaged Skill     Metadata, usage instructions, examples, basic happy-path tests
Level 2 — Verified Skill  ←  Typed boundaries, behavioral tests, token budgets, security posture  ← we're moving here
Level 3 — Governed Capability  Runtime policy, sandboxing, human approval gates, full audit trail
```

---

## ✏️ Explore Further

The cell below has an alternative brief pre-filled. Run it as-is, or replace with your own profile data and observe:
- How token counts change with different input lengths
- Whether the tone signals match what you'd expect for your chosen tone
- Which platforms fail the character limit test most often

In [ ]:
# ── Try it yourself ──────────────────────────────────────────────────────────
# Replace any field below with your own profile data.
# ─────────────────────────────────────────────────────────────────────────────
MY_BRIEF = {
    "name": "Jordan Kim",
    "what_you_do": "Platform engineer focused on internal developer tooling and CI/CD automation",
    "core_belief": "Fast feedback loops make better engineers",
    "personal_texture": ["mechanical keyboards", "distance cycling", "sci-fi reader"],
    "tone": "warm & witty",
    "cta": "Let's talk about developer experience",
    "links": ["https://github.com/jordankim"],
}

for platform in ["Twitter/X", "LinkedIn", "GitHub"]:
    result = ask(build_prompt(MY_BRIEF, [platform]))
    raw  = result["output"].strip()
    bio  = raw.split(":", 1)[-1].strip() if ":" in raw else raw
    print(f"[{platform}] ({len(bio)} chars)")
    print(bio)
    print()

---

## What's next

This notebook covered the three gaps and their fixes. **Part 2 of this series** goes deeper on what a production-grade skill contract actually looks like — behavioral boundaries, capability manifests, and resource budgets that put real numbers on what "resourceful" means.

→ **[Read the full series: How Resourceful Is Your AI Skill?](https://sriharshacr.github.io/blogs/how-resourceful-is-your-ai-skill/)**

---

*The `profile-bio` skill used as the subject of this notebook is open source: [github.com/SriharshaCR/open-skills/tree/main/profile-bio](https://github.com/SriharshaCR/open-skills/tree/main/profile-bio)*